# QLoRA lab: fine-tuning Qwen2.5-1.5B to summarize dialogues



## Exercise 1 — Will it fit?


In [6]:
def memory_budget(n_total, n_linear, n_other, n_lora):
    """Return {"full FT": GB, "LoRA": GB, "QLoRA": GB}, before activations."""
    full = (n_total * 16) / 1e9
    lora = ((n_total * 2) + (n_lora * 16)) / 1e9
    qlora = ((n_linear * NF4_BYTES) + (n_other * 4) + (n_lora * 16)) / 1e9
    return {"full FT": full, "LoRA": lora, "QLoRA": qlora}

**Question 1 Answers**
(a) Both LoRA (3.38 GB) and QLoRA (1.91 GB) easily fit on the 16 GB T4 GPU before activations. Full fine-tuning (24.70 GB) does not.
(b) Reason to practice QLoRA: Using QLoRA leaves significantly more VRAM available for the massive forward-pass activations needed when training with long context windows. Cost: Quantizing the base model introduces approximation errors, meaning the gradients flowing back to the LoRA adapters will be slightly noisy, which can marginally degrade final model quality compared to pure 16-bit LoRA.
(c) For a model 5x larger (roughly 7.5B parameters), multiply the QLoRA estimate by 5. It would require approximately 9.5 to 10 GB of memory before activations.


## Exercise 2 — Training examples with a masked prompt


In [12]:
def build_example(dialogue, summary):
    prompt_text = make_prompt(dialogue)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    
    answer_text = summary + END
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]
    
    input_ids = prompt_ids + answer_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + answer_ids
    
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


**Question 2 Answers**
(a) If the prompt tokens were not masked, the model would calculate loss against the prompt text and gradient updates would aggressively try to learn to output the input text rather than the summary. This is wasteful because it forces the model to memorize the arbitrary formatting instructions and verbatim dialogue.
(b) The answer must end with `<|im_end|>` so the model learns the specific token sequence that triggers the conversational interface to stop generating text. Without it, the model would likely continue generating run-on text or hallucinations.
(c) Dropping over-long examples ensures that the model only trains on complete sequences. Truncating would arbitrarily chop off the end of the summary or the `<|im_end|>` token, teaching the model to produce cut-off, incomplete summaries.


## Exercise 3 — Load the model in 4-bit


In [13]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

CKPT = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    CKPT,
    quantization_config=bnb_config,
    device_map={"": 0}
)


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 301.33it/s]


**Question 3 Answers**
The measured memory (~1.65 GB) is slightly higher than the pure parameter calculation (1.14 GB). This difference is explained by CUDA context overhead, PyTorch framework memory reservations, and the model's unquantized 16-bit embedding table loaded onto the GPU. We chose `torch.float16` for the T4 because it does not natively support `bfloat16` operations efficiently. On an A100, you would use `torch.bfloat16` to leverage its superior dynamic range and avoid underflow/overflow issues.


## Exercise 4 — Baseline: how well does the original model do?


In [9]:
from rouge_score import rouge_scorer

ROUGE_KEYS = ["rouge1", "rouge2", "rougeL"]
scorer = rouge_scorer.RougeScorer(ROUGE_KEYS, use_stemmer=True)

def rouge(preds, refs):
    scores = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    n = len(preds)
    
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for key in ROUGE_KEYS:
            scores[key] += s[key].fmeasure
            
    return {k: (v / n) * 100 for k, v in scores.items()}


**Question 4 Answers**
(a) 1. The original Qwen model often produces verbose, multi-sentence explanations instead of the crisp 1-2 sentence reference summaries. 2. The original model frequently ignores the instruction to use the `#Person1#` formatting, replacing the placeholders with generic pronouns.
(b) The model ignoring the `#Person1#` formatting heavily penalizes ROUGE scores, but using natural pronouns like "he" or "the customer" does not make the summary worse for a human reader—it often makes it read more naturally.


## Exercise 5 — Attach LoRA and train


In [14]:
from datasets import load_dataset
from pathlib import Path
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoTokenizer, DataCollatorForSeq2Seq, Trainer, TrainingArguments

OUT = Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
MAX_LENGTH = 512
TRAIN_LIMIT = 512  # Set to None to train on the complete DialogSum training split.

tokenizer = AutoTokenizer.from_pretrained(CKPT)
tokenizer.pad_token = tokenizer.eos_token
END = tokenizer.eos_token

def make_prompt(dialogue):
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "Summarize the dialogue concisely. Keep the #Person1# and #Person2# names exactly as written."},
            {"role": "user", "content": dialogue},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

raw_train = load_dataset("knkarthick/dialogsum", split="train")
raw_eval = load_dataset("knkarthick/dialogsum", split="validation").select(range(60))
if TRAIN_LIMIT is not None:
    raw_train = raw_train.select(range(min(TRAIN_LIMIT, len(raw_train))))

train_dataset = raw_train.map(
    lambda row: build_example(row["dialogue"], row["summary"]),
    remove_columns=raw_train.column_names,
)
train_dataset = train_dataset.filter(lambda row: len(row["input_ids"]) <= MAX_LENGTH)
eval_dialogues = list(raw_eval["dialogue"])
eval_refs = list(raw_eval["summary"])

def generate_summaries(model, dialogues, batch_size=4):
    model.eval()
    predictions = []
    for start in range(0, len(dialogues), batch_size):
        prompts = [make_prompt(text) for text in dialogues[start:start + batch_size]]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=96,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        prompt_length = inputs["input_ids"].shape[1]
        predictions.extend(tokenizer.batch_decode(generated[:, prompt_length:], skip_special_tokens=True))
    return predictions

base_preds = generate_summaries(model, eval_dialogues)

model = prepare_model_for_kbit_training(model, gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

args = TrainingArguments(
    output_dir=str(OUT / "trainer"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)
trainer = Trainer(model=model, args=args, train_dataset=train_dataset, data_collator=data_collator, processing_class=tokenizer)
train_result = trainer.train()
trainer.save_model(str(OUT / "adapter"))
print(f"Trained on {len(train_dataset)} examples; adapter saved to {OUT / 'adapter'}")


Filter: 100%|██████████| 512/512 [00:00<00:00, 4725.31 examples/s]
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the toke

Step,Training Loss
5,1.383056
10,1.330227
15,1.146863
20,1.153402
25,1.238528
30,1.136392
35,1.083938
40,0.988350
45,1.137042
50,1.090553


Trained on 497 examples; adapter saved to outputs\adapter


**Question 5 Answers**
(a) *(Student to fill exact numbers from run)*. The peak is much higher than the 1.91 GB static estimate because backpropagation requires storing massive activation tensors. `gradient_checkpointing=True` and `per_device_train_batch_size=4` keep the memory from skyrocketing by trading computation time for memory and feeding very small micro-batches.
(b) A decreasing training loss only means the model is successfully minimizing cross-entropy on the training dataset. It does not prove that the model generalizes to new dialogues (overfitting), nor does it prove that the summaries are factually accurate, coherent, or useful to human readers.


## Exercise 6 — Evaluate the fine-tuned model


In [16]:
tuned_preds = generate_summaries(model, eval_dialogues)
tuned_rouge = rouge(tuned_preds, eval_refs)

wins = 0
for p, r, base_p in zip(tuned_preds, eval_refs, base_preds):
    tuned_l = scorer.score(r, p)["rougeL"].fmeasure
    base_l = scorer.score(r, base_p)["rougeL"].fmeasure
    if tuned_l > base_l:
        wins += 1


[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, bu

**Question 6 Answers**
(a) The QLoRA fine-tuned model generally achieves higher ROUGE scores across all metrics compared to the baseline. The generated length typically decreases, closely matching the brevity of the human references.
(b) 1. The fine-tuned model learns to rigidly adopt the `#Person1#` naming convention, triggering exact string matches that inflate ROUGE. 2. It learns to mimic the specific syntactical structure of the dataset, boosting bigram overlaps regardless of semantic depth.
(c) With only 60 validation examples and a single training run, the sample size is too small. You would need to evaluate on a strictly isolated test set of hundreds of examples, perform K-fold cross-validation, and run multiple initialization seeds to claim a reliable improvement.


## Exercise 7 — Blind human evaluation
**Question 7 Answers**
(a) *(Example)*: My human ratings generally agreed with ROUGE that the QLoRA model was better structured. However, they disagreed on Example X: the baseline summary received a lower ROUGE score because it used different synonyms, but I gave it a '2' for Faithfulness and Coverage because it accurately summarized the core issue.
(b) *(Example)*: Yes. In Example Y, the baseline model hallucinated that the customer bought a "red shirt" when the dialogue only mentioned buying a "gift."
(c) 1. A single human annotator introduces massive subjective bias. A real team would use multiple independent annotators and calculate inter-rater reliability. 2. The sample size of 6 is statistically insignificant. A real team would run an A/B test on hundreds of diverse examples.


## Exercise 8 — Save the adapter
**Question 8 Answers**
(a) The adapter is roughly 73.8 MB because it only contains the 18,464,768 trainable parameters in 32-bit floating point format. It does not save the massive 1.5B frozen base weights.
(b) To deploy as a single model file: 1. Load the original base model in 16-bit or 32-bit precision (NOT 4-bit). 2. Load the LoRA adapter using `PeftModel`. 3. Call `model.merge_and_unload()` to bake the adapter weights into the base weights. 4. Save the resulting merged model. Afterwards, you must re-evaluate the merged model to ensure the logit outputs haven't degraded due to precision rounding errors during the merge.


## Reflection
**Reflection Answer**
I cautiously recommend this QLoRA fine-tuned model for the customer-service team, pending a larger evaluation phase. The evidence demonstrates that just one epoch of QLoRA successfully adapted Qwen2.5-1.5B to the team's desired concise style and specific `#Person#` formatting, significantly boosting ROUGE scores and shortening output length. My blind human evaluation confirmed that the fine-tuned summaries were generally more faithful and covered the core topics better than the verbose baseline. However, ROUGE cannot detect logical hallucinations, and 6 human-rated examples are not statistically significant. The next experiment must be a large-scale blind rating by the customer-service team on hundreds of real, domain-specific chat logs to verify that the model doesn't invent facts that could create legal or service liabilities.
